# H6. Actions change the future
Book: Reinforcement Learning, Bellman Principle of Optimality.

Supplement: Alisa's math notes, [Markov chains](https://alisawuffles.notion.site/math-notes#3737eb873605809498dbf3714e119c5c) and [First-step analysis](https://alisawuffles.notion.site/math-notes#3737eb8736058061ac5bd60c74526bb8). Use the corrected recurrence in the course reading cautions.
These selected readings are optional support. The classroom examples define the required scope.
Reading correction: for two consecutive heads, count the successful final flip.
E0 = 1 + E0/2 + E1/2 and E1 = 1 + E0/2, so E0 = 6.

We know this toy environment's transitions. Solving it is planning.
RL studies learning to act when relevant quantities must be learned from experience.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def machine_simulation(policy="plan", p=.1, trials=5000, horizon=2, seed=600, gamma=1.0):
    if horizon not in (1, 2):
        raise ValueError("This classroom experiment uses one or two periods.")
    if policy not in ("plan", "greedy", "service-first") or not 0 <= p <= 1:
        raise ValueError("Choose a supported policy and a probability in [0, 1].")
    if not 0 <= gamma <= 1:
        raise ValueError("Choose a discount factor in [0, 1].")
    rng = np.random.default_rng(seed)
    working = np.ones(trials, dtype=bool)
    total = np.zeros(trials)
    for t in range(horizon):
        service = horizon-t == 2 and (policy == "service-first" or
                                      (policy == "plan" and 1+2*gamma > 2+2*gamma*p))
        if service:
            total += gamma**t * working.astype(float)
            working[:] = True
        else:
            total += gamma**t * 2*working
            working = np.where(working, rng.random(trials) < p, True)
    return total

## Environment
Two periods, initial state Working. Operate earns 2 now and leaves the machine
working with probability p. Service earns 1 now and keeps it working.
In Broken, repair earns 0 and restores Working for the next period.
Rewards after the second period do not count.

## A. Immediate reward versus total reward (30 minutes)
First five minutes: under always Operate/Repair, predict the state distribution
after one and two transitions. Rows are current states; columns are next states.
Does the Markov property mean adjacent states are independent?

In [ ]:
transition = np.array([[.1, .9], [1., 0.]])
distribution = np.array([1., 0.])
print("One step:", distribution @ transition)
print("Two steps:", distribution @ transition @ transition)
assert np.allclose(transition.sum(axis=1), 1)
assert np.allclose(distribution @ transition @ transition, [.91, .09])

Calculate the final-period values first. Predict mean total reward for each policy.

In [ ]:
p = .1
trials = 5000
greedy = machine_simulation("greedy", p=p, trials=trials)
planned = machine_simulation("service-first", p=p, trials=trials)
print("Operate first: simulation", greedy.mean(), "exact", 2+2*p)
print("Service first: simulation", planned.mean(), "exact", 3.)
fig, ax = plt.subplots()
ax.hist(greedy, bins=[1.5, 2.5, 3.5, 4.5], density=True, alpha=.6, label="Greedy")
ax.hist(planned, bins=[1.5, 2.5, 3.5, 4.5], density=True, alpha=.6, label="Service first")
ax.set(xlabel="Total reward in two periods", ylabel="Relative frequency")
ax.legend()
plt.show()

A lucky greedy trajectory can outperform the planned policy.
Explain why one run cannot establish which policy has higher expected return.

Prediction:

Observation:

Explanation:

## B. When does the preferred action change? (30 minutes)
Solve 2+2p=3 before changing p. Predict the answer if there is only one period.

In [ ]:
probabilities = np.linspace(0, 1, 11)
means = [machine_simulation("greedy", p=float(p), trials=10000).mean()
         for p in probabilities]
plt.plot(probabilities, 2+2*probabilities, label="Operate first: exact")
plt.scatter(probabilities, means, label="Operate first: simulation")
plt.axhline(3, color="black", label="Service first")
plt.xlabel("Probability of staying Working after operating")
plt.ylabel("Expected two-period reward")
plt.legend()
plt.show()
print("One-period planned reward:", machine_simulation("plan", horizon=1).mean())

Last five minutes of B: keep p=.1 and change gamma from 1 to .5.
Predict which first action maximizes E[R1 + gamma*R2].

In [ ]:
for discount in [1., .5]:
    reliability = .1
    exact_operate = 2+2*discount*reliability
    exact_service = 1+2*discount
    simulated = machine_simulation("plan", p=reliability, gamma=discount, trials=100000)
    print("gamma", discount, "operate", exact_operate, "service", exact_service,
          "planned simulation", simulated.mean())
    assert abs(simulated.mean()-max(exact_operate, exact_service)) < .02

Explain why the policy depends on time remaining as well as physical state.
What would have to be estimated if p were unknown?

Prediction:

Observation:

Explanation:

## Individual exit
At p=0.8, calculate both first-action values. Choose an action and justify it.
Distinguish a predicted outcome from a policy that chooses actions.